DAG CUSTOM EVAL

In [11]:
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import (
    BinaryJudgementNode,
    DeepAcyclicGraph,
    NonBinaryJudgementNode,
    TaskNode,
    VerdictNode,
)
from deepeval.test_case import SingleTurnParams

BUG FIXES

In [16]:
import deepeval.metrics.base_metric as _base_metric
from deepeval.metrics.dag.schema import TaskNodeOutput
from pydantic import ValidationError


_original_resolve = _base_metric.resolve_template

def _patched_resolve(feature, class_name, method, **kwargs):
    if class_name == "BinaryJudgement" and method == "generate_non_binary_verdict":
        class_name = "NonBinaryJudgement"  # the real fix -- should never have said "Binary"
    return _original_resolve(feature, class_name, method, **kwargs)

_base_metric.resolve_template = _patched_resolve



## bug 2
_original_validate = TaskNodeOutput.model_validate.__func__

def _patched_validate(cls, obj, *args, **kwargs):
    try:
        return _original_validate(cls, obj, *args, **kwargs)
    except ValidationError:
        if isinstance(obj, dict) and isinstance(obj.get("output"), dict) and len(obj["output"]) == 1:
            unwrapped = {**obj, "output": next(iter(obj["output"].values()))}
            return _original_validate(cls, unwrapped, *args, **kwargs)
        raise

TaskNodeOutput.model_validate = classmethod(_patched_validate)


print("Patched.")

Patched.


In [17]:
def build_dag_metric():
    correct_order_node = NonBinaryJudgementNode(
        criteria="Are the summary headings in the correct order: 'intro' => 'body' => 'conclusion'?",
        children=[
            VerdictNode(verdict="Yes", score=10),
            VerdictNode(verdict="Two are out of order", score=4),
            VerdictNode(verdict="All out of order", score=2),
        ],
    )

    correct_headings_node = BinaryJudgementNode(
        criteria="Does the summary headings contain all three: 'intro', 'body', and 'conclusion'?",
        children=[
            VerdictNode(verdict=False, score=0),
            VerdictNode(verdict=True, child=correct_order_node),
        ],
    )

    extract_headings_node = TaskNode(
        instructions="Extract all headings in `actual_output`",
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
        output_label="Summary headings",
        children=[correct_headings_node, correct_order_node],
    )

    dag = DeepAcyclicGraph(root_nodes=[extract_headings_node])
    return DAGMetric(name="Format Correctness", dag=dag, model=judge, async_mode=False)


In [18]:
meeting_transcript = (
    'Alice: "Today\'s agenda: product update, blockers, and marketing timeline. Bob, updates?"\n'
    'Bob: "Core features are done, but we\'re optimizing performance for large datasets. Fixes '
    'by Friday, testing next week."\n'
    'Alice: "Charlie, does this timeline work for marketing?"\n'
    'Charlie: "We need finalized messaging by Monday."\n'
    'Alice: "Bob, can we provide a stable version by then?"\n'
    'Bob: "Yes, we\'ll share an early build."\n'
    'Charlie: "Great, we\'ll start preparing assets."\n'
    'Alice: "Plan: fixes by Friday, marketing prep Monday, sync next Wednesday. Thanks, everyone!"'
)


In [19]:
from deepeval.test_case import LLMTestCase

dag_test_cases = [
    LLMTestCase(
        input=meeting_transcript,
        actual_output=(
            "Intro:\nAlice outlined the agenda: product updates, blockers, and marketing "
            "alignment.\n\nBody:\nBob reported performance issues being optimized, with fixes "
            "expected by Friday. Charlie requested finalized messaging by Monday for marketing "
            "preparation. Bob confirmed an early stable build would be ready.\n\nConclusion:\n"
            "The team aligned on next steps: engineering finalizing fixes, marketing preparing "
            "content, and a follow-up sync scheduled for Wednesday."
        ),
    ),
    LLMTestCase(
        input=meeting_transcript,
        actual_output=(
            "Intro:\nAlice outlined the agenda.\n\nBody:\nBob reported fixes by Friday and "
            "Charlie needs messaging by Monday."
        ),  # no Conclusion heading -- the gate should fail and skip the order check
    ),
]

RUN DAG AND INSPECT THE TRACE

In [20]:
for i, tc in enumerate(dag_test_cases, start=1):
    metric = build_dag_metric()
    metric.measure(tc)
    verdict = "PASS" if metric.is_successful() else "FAIL"
    raw_score = round(metric.score * 10)
    print(f"=== Test case {i}: [{verdict}] raw {raw_score}/10 -> normalized {metric.score:.2f} ===")
    print(metric.verbose_logs)
    print()

NameError: name 'judge' is not defined